In [10]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')   

In [4]:
review = pd.read_csv('./data/review.csv')
print(review.shape)
review.head(2)

(2050, 7)


,date,keyword,title,contents,comments,board_titles,url
0,2025-05-02- 11:57,진주성,‘제24회 진주논개제’ 5월 3~6일 진주성과 진주대첩 역사공원 일원에서!! (일정...,‘제24회 진주논개제’ 5월 3~6일 진주성과 진주대첩 역사공원 일원에서!! (일정...,댓글 없음,진주인터넷뉴스\n‘제24회 진주논개제’ 5월 3~6일 진주성과 진주대첩 역사공원 일...,https://cafe.naver.com/ca-fe/ArticleRead.nhn?c...
1,2025-05-02- 10:43,진주성,진주성 인근 브런치 카페 / 카페피플,https://m.blog.naver.com/mukkebiyo/22385306891...,댓글 없음,블로그/유튜브/댓가성 후기\n진주성 인근 브런치 카페 / 카페피플,https://cafe.naver.com/ca-fe/ArticleRead.nhn?c...


In [5]:
review['contents']

0       ‘제24회 진주논개제’ 5월 3~6일 진주성과 진주대첩 역사공원 일원에서!! (일정...
1       https://m.blog.naver.com/mukkebiyo/22385306891...
2       ​진주성 #논개제 에 로컬 장터가 찾아온다한복과 패랭이 모자를 쓴 현대판 장돌뱅이들...
3       https://m.blog.naver.com/mukkebiyo/22384025292...
4       곱배기는 면 두덩이~아들냄 다 못먹네요~욕심부리더니..시원하게  잘먹었습니다~♡​ ...
                              ...                        
2045    [이미지: https://ssl.pstatic.net/static/cafe/cafe...
2046    [이미지: https://blogpfthumb-phinf.pstatic.net/da...
2047    근처 지나가는데 빵냄새가 솔솔~눈 잠깐 깜빡였는데 벌써 입구 앞🤣내부는 화이트+블랙...
2048    [이미지: https://ssl.pstatic.net/static/cafe/cafe...
2049    [이미지: https://blogpfthumb-phinf.pstatic.net/Mj...
Name: contents, Length: 2050, dtype: object

In [7]:
from transformers import pipeline

# KLUE RoBERTa-base를 한국어 감성분석용으로 파인튜닝해 둔 공개 모델
classifier = pipeline(
    task="sentiment-analysis",
    model="Chamsol/klue-roberta-sentiment-classification",  # GPU를 쓰려면 device=0 추가
    tokenizer="klue/roberta-base"
)

texts = ["이 영화 정말 최고였어요!", "내용이 너무 지루하고 길었어."]
print(classifier(texts))

model.safetensors:  52%|#####2    | 703M/1.35G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/752k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

Device set to use mps:0


[{'label': 'LABEL_1', 'score': 0.9899641871452332}, {'label': 'LABEL_0', 'score': 0.9930329322814941}]


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers.pipelines.text_classification import TextClassificationPipeline
import pandas as pd

# 1) review DataFrame 불러오기
# 예: review = pd.read_csv('reviews.csv')

# 2) 모델·토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained("klue/roberta-base")
model = AutoModelForSequenceClassification.from_pretrained(
    "Chamsol/klue-roberta-sentiment-classification"
)

# 3) 텍스트 전용 파이프라인 생성 (torchvision 의존성 없음)
classifier = TextClassificationPipeline(
    model=model,
    tokenizer=tokenizer,
    device=-1,            # CPU에서 실행할 때는 -1, GPU를 쓰려면 0 등으로 설정
    return_all_scores=False,
    function_to_apply="softmax"
)

# 4) 배치 단위로 감성분석 수행
texts = review['contents'].astype(str).tolist()
labels, scores = [], []

batch_size = 8
for i in range(0, len(texts), batch_size):
    batch_texts = texts[i:i+batch_size]
    # 긴 텍스트는 모델 max_length에 맞춰 자르도록 truncation=True
    results = classifier(batch_texts, truncation=True)
    for r in results:
        labels.append(r['label'])
        scores.append(r['score'])

# 5) 결과를 DataFrame에 추가
review['sentiment_label'] = labels
review['sentiment_score'] = scores

# 6) 확인
review.head(2)

Device set to use cpu


,date,keyword,title,contents,comments,board_titles,url,sentiment_label,sentiment_score
0,2025-05-02- 11:57,진주성,‘제24회 진주논개제’ 5월 3~6일 진주성과 진주대첩 역사공원 일원에서!! (일정...,‘제24회 진주논개제’ 5월 3~6일 진주성과 진주대첩 역사공원 일원에서!! (일정...,댓글 없음,진주인터넷뉴스\n‘제24회 진주논개제’ 5월 3~6일 진주성과 진주대첩 역사공원 일...,https://cafe.naver.com/ca-fe/ArticleRead.nhn?c...,LABEL_1,0.738082
1,2025-05-02- 10:43,진주성,진주성 인근 브런치 카페 / 카페피플,https://m.blog.naver.com/mukkebiyo/22385306891...,댓글 없음,블로그/유튜브/댓가성 후기\n진주성 인근 브런치 카페 / 카페피플,https://cafe.naver.com/ca-fe/ArticleRead.nhn?c...,LABEL_1,0.959217


In [13]:
review['keyword'] = review['keyword'].str.replace(
    '경상국립대',
    '경상대'
)

In [17]:
sent = review.groupby('keyword', as_index=False)['sentiment_score'].mean()
sent

,keyword,sentiment_score
0,CGV,0.726648
1,LH,0.711766
2,갤러리아,0.752768
3,경상대,0.736908
4,남강,0.726113
5,롯데마트,0.676764
6,롯데몰,0.700681
7,롯데시네마,0.669798
8,메가박스,0.688804
9,모다아울렛,0.715585


In [ ]:
location_data = {
    "CGV": {"address": "경상남도 진주시 에나로127번길 30", "latitude": 35.180660, "longitude": 128.143954},
    "LH": {"address": "경상남도 진주시 충의로 19", "latitude": 35.180660, "longitude": 128.143954},
    "갤러리아": {"address": "경상남도 진주시 진주대로 1095", "latitude": 35.180660, "longitude": 128.143954},
    "경상대": {"address": "경상남도 진주시 진주대로 501", "latitude": 35.152120, "longitude": 128.104720},
    "남강": {"address": "경상남도 진주시 남강로 626", "latitude": 35.189376, "longitude": 128.082690},
    "롯데마트": {"address": "경상남도 진주시 동진로 440", "latitude": 35.180660, "longitude": 128.143954},
    "롯데몰": {"address": "경상남도 진주시 동진로 440", "latitude": 35.180660, "longitude": 128.143954},
    "롯데시네마": {"address": "경상남도 진주시 동진로 440", "latitude": 35.180660, "longitude": 128.143954},
    "메가박스": {"address": "경상남도 진주시 진양호로 521", "latitude": 35.180660, "longitude": 128.143954},
    "모다아울렛": {"address": "경상남도 진주시 정촌면 삼일로105번길 15", "latitude": 35.180660, "longitude": 128.143954},
    "엠비씨네": {"address": "경상남도 진주시 평거대로 22", "latitude": 35.180660, "longitude": 128.143954},
    "이마트": {"address": "경상남도 진주시 진양호로 477", "latitude": 35.180660, "longitude": 128.143954},
    "중앙시장": {"address": "경상남도 진주시 진양호로547번길 8-1", "latitude": 35.180660, "longitude": 128.143954},
    "진주성": {"address": "경상남도 진주시 남강로 626", "latitude": 35.189376, "longitude": 128.082690},
    "진주역": {"address": "경상남도 진주시 진주역로 50", "latitude": 35.180660, "longitude": 128.143954},
    "칠암캠": {"address": "경상남도 진주시 동진로 33", "latitude": 35.179100, "longitude": 128.094839},
    "탑마트": {"address": "경상남도 진주시 평거로 70", "latitude": 35.180660, "longitude": 128.143954},
    "홈플러스": {"address": "경상남도 진주시 진주대로 1095", "latitude": 35.180660, "longitude": 128.143954}
}

# 주소, 위도, 경도 컬럼 추가
sent['address'] = sent['keyword'].map(lambda x: location_data[x]['address'])
sent['latitude'] = sent['keyword'].map(lambda x: location_data[x]['latitude'])
sent['longitude'] = sent['keyword'].map(lambda x: location_data[x]['longitude'])

sent

,keyword,sentiment_score,address,latitude,longitude
0,CGV,0.726648,경상남도 진주시 에나로127번길 30,35.180660,128.143954
1,LH,0.711766,경상남도 진주시 사들로 19,35.180660,128.143954
2,갤러리아,0.752768,경상남도 진주시 진주대로 1095,35.180660,128.143954
3,경상대,0.736908,경상남도 진주시 진주대로 501,35.152120,128.104720
4,남강,0.726113,경상남도 진주시 남강로 626,35.189376,128.082690
5,롯데마트,0.676764,경상남도 진주시 동진로 440,35.180660,128.143954
6,롯데몰,0.700681,경상남도 진주시 동진로 440,35.180660,128.143954
7,롯데시네마,0.669798,경상남도 진주시 동진로 440,35.180660,128.143954
8,메가박스,0.688804,경상남도 진주시 진양호로 521,35.180660,128.143954
9,모다아울렛,0.715585,경상남도 진주시 정촌면 삼일로105번길 15,35.180660,128.143954
